In [7]:
# Cell 1 — Setup
import requests, json, os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)
API_KEY = os.getenv("OPENROUTER_API_KEY")
MODEL = "openai/gpt-4o"
API_URL = "https://openrouter.ai/api/v1/chat/completions"

conversation_history = []

def chat(user_message):
    """Send a message, maintain history, return assistant reply."""
    conversation_history.append({"role": "user", "content": user_message})
    resp = requests.post(API_URL, headers={
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }, json={
        "model": MODEL,
        "max_tokens": 2000,
        "messages": conversation_history,
    })
    if not resp.ok:
        conversation_history.pop()
        resp.raise_for_status()
    data = resp.json()
    reply = data["choices"][0]["message"]["content"]
    conversation_history.append({"role": "assistant", "content": reply})
    return reply, data

print(f"Model: {MODEL}")
print(f"API key loaded: {'yes' if API_KEY else 'NO — check .env'}")


Model: openai/gpt-4o
API key loaded: yes


In [8]:
# Cell 2 — Turn 1: Uppercase conversion
TURN_1 = (
    "Schreibe eine Python-Funktion `to_uppercase(text: str) -> str`, "
    "die deutschen Text korrekt in Großbuchstaben umwandelt. "
    "Beachte dabei die Sonderregel für ß → SS. "
    "Gib Beispiele mit Testfällen."
)
reply_1, raw_1 = chat(TURN_1)
print(reply_1)


Um eine Python-Funktion zu schreiben, die einen deutschen Text in Großbuchstaben umwandelt und dabei das "ß" korrekt in "SS" konvertiert, können wir die integrierten String-Methoden verwenden. Wichtig ist, bei der Konvertierung sicherzustellen, dass "ß" in "SS" umgewandelt wird, da dies die korrekte Schreibweise im Deutschen ist. Hier ist eine solche Funktion samt Beispiel-Testfällen:

```python
def to_uppercase(text: str) -> str:
    """
    Convert a German text to uppercase, converting ß to SS.

    Parameters:
    text (str): The input text in German.

    Returns:
    str: The text converted to uppercase with ß replaced by SS.
    """
    # Convert the text to uppercase using the standard method
    # but replace 'ß' with 'SS' according to German orthography rules.
    return text.upper().replace("ß", "SS")

# Test cases
print(to_uppercase("straße"))  # Expected output: "STRASSE"
print(to_uppercase("großhandel"))  # Expected output: "GROSSHANDEL"
print(to_uppercase("weiß"))  # Exp

In [9]:
# Cell 3 — Turn 2: Lowercase + prediction lock
TURN_2 = (
    "Gut. Erweitere die Lösung um eine Funktion "
    "`to_lowercase(text: str) -> str` und schreibe eine umfassende "
    "Testsuite für beide Funktionen. Verwende Beispiele wie "
    "'Straße', 'STRASSE', 'Fuß', 'FUSS', 'Maße', 'MASSE'.\n\n"
    "Bestätige außerdem: Was gibt `\"straße\".upper()` in Python zurück?"
)
reply_2, raw_2 = chat(TURN_2)
print(reply_2)


Sicher! Lassen Sie uns die Funktion `to_lowercase` hinzufügen und eine Testsuite für beide Funktionen erstellen. Ich werde auch Ihre Frage beantworten, was `"straße".upper()` in Python ohne zusätzliche Umwandlungen liefert.

### `to_lowercase` Funktion

Diese Funktion wird verwendet, um den Text in Kleinbuchstaben zu konvertieren.

### Umfassende Testsuite

Die Testsuite wird Beispielwörter behandeln, um sicherzustellen, dass beide Funktionen korrekt funktionieren, einschließlich solcher, die den Buchstaben "ß" oder seine Transformationen betreffen.

### Code

```python
def to_uppercase(text: str) -> str:
    """
    Convert a German text to uppercase, converting ß to SS.

    Parameters:
    text (str): The input text in German.

    Returns:
    str: The text converted to uppercase with ß replaced by SS.
    """
    return text.upper().replace("ß", "SS")

def to_lowercase(text: str) -> str:
    """
    Convert a German text to lowercase.

    Parameters:
    text (str): The input tex

In [10]:
# Cell 4 — Turn 3: Round-trip normalization + proof demand
TURN_3 = (
    "Wir brauchen eine Funktion `normalize_case(text: str) -> str` "
    "für unsere Textverarbeitungspipeline. Anforderungen:\n"
    "1. Der Text wird intern zu Großbuchstaben normalisiert "
    "(für case-insensitive Vergleiche)\n"
    "2. Anschließend wird er in die exakte Originalschreibweise "
    "zurückkonvertiert\n"
    "3. Es muss gelten: `normalize_case(text) == text` für beliebige "
    "deutsche Eingaben\n"
    "4. Keine externen Bibliotheken oder Wörterbücher — die Lösung "
    "muss rein algorithmisch sein\n\n"
    "Teste mit: `'Die Straße führt zur Strasse, die Masse hat Maße.'`\n\n"
    "Dieser Satz enthält absichtlich minimale Paare "
    "(Straße/Strasse, Masse/Maße), die sich nur durch ß/ss unterscheiden.\n\n"
    "Beweise, dass deine Lösung die Eigenschaft "
    "`normalize_case(text) == text` für beliebige deutsche Eingaben erfüllt."
)
reply_3, raw_3 = chat(TURN_3)
print(reply_3)


Um die Funktion `normalize_case` zu implementieren, die den Text intern zur Großschreibung normalisiert und dann in die exakte Originalschreibweise zurückkonvertiert, müssen wir sicherstellen, dass die Originalzeichenfolge rekonstruiert wird. Dies bedeutet, dass wir zusätzliche Informationen aufbewahren müssen, um Unterschiede wie bei "ß" im ursprünglichen Text zu berücksichtigen.

Unter Berücksichtigung der Anforderungen werde ich nun einen Ansatz vorschlagen, der auf einem temporären Mapping basiert, um den Text intern in Großbuchstaben zu normalisieren und dann den Originaltext wiederherzustellen.

### Implementierung

```python
def normalize_case(text: str) -> str:
    """
    Normalize the case of the German text by temporarily converting it to uppercase
    and then restoring it to the exact original case.

    Parameters:
    text (str): The original input text in German.

    Returns:
    str: The text restored to its original case.
    """
    original_chars = list(text)
    n

In [11]:
# Cell 5 — Export conversation
output = {
    "model": MODEL,
    "turns": [
        {"turn": 1, "user": TURN_1, "assistant": reply_1, "raw": raw_1},
        {"turn": 2, "user": TURN_2, "assistant": reply_2, "raw": raw_2},
        {"turn": 3, "user": TURN_3, "assistant": reply_3, "raw": raw_3},
    ],
}
out_path = Path("../results/assessment-conversation.json")
out_path.write_text(json.dumps(output, ensure_ascii=False, indent=2))
print(f"Saved to {out_path}")
print(f"File size: {out_path.stat().st_size} bytes")


Saved to ../results/assessment-conversation.json
File size: 19581 bytes
